# Train Baseline Reward Model — Google Colab

**Before running:** set runtime to GPU via `Runtime → Change runtime type → T4 GPU`.

This notebook:
1. Verifies the GPU
2. Installs dependencies
3. Clones the repo
4. Trains the reward model (~10 min on T4)
5. Saves the model to Google Drive

## 1. Check GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found — go to Runtime → Change runtime type → T4 GPU")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Install dependencies

In [ ]:
%pip install -q \
    "transformers>=5.0.0" \
    "trl>=1.5.0" \
    "datasets>=2.14.0" \
    "peft>=0.10.0" \
    "accelerate>=0.27.0"

## 3. Clone the repo\n\nThis repo is private. Before running this cell, add your GitHub token to Colab Secrets:\n- Left sidebar → 🔑 **Secrets** → **Add new secret**\n- Name: `GITHUB_TOKEN`, Value: a token with `repo` scope ([create one here](https://github.com/settings/tokens/new?scopes=repo))\n- Toggle **Notebook access** on

In [ ]:
import os, subprocess, sys

REPO_DIR = "/content/rlhf-bias-decomp"

# Load token from Colab Secrets, falling back to a manual prompt
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    print("Token loaded from Colab Secrets.")
except Exception:
    import getpass
    token = getpass.getpass("Paste your GitHub token (input is hidden): ")

REPO_URL = f"https://{token}@github.com/moisheu/rlhf-bias-decomp.git"

if os.path.exists(REPO_DIR):
    print("Repo already cloned — pulling latest...")
    result = subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True, text=True)
else:
    print("Cloning repo...")
    result = subprocess.run(["git", "clone", REPO_URL, REPO_DIR], capture_output=True, text=True)

# Strip token from output before printing
output = (result.stdout or result.stderr).replace(token, "<TOKEN>")
print(output)

if not os.path.exists(REPO_DIR):
    print("Clone failed — check that your token has 'repo' scope.")
    sys.exit(1)

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
print("Contents:", os.listdir("."))

## 4. Train the reward model

Expected time: ~10 min on T4. Loss should start around 0.69 and decrease.

In [ ]:
!python -m src.train_reward_model

## 5. Verify the saved model

In [ ]:
import os

model_dir = "results/reward_model_baseline"
files = os.listdir(model_dir)
print(f"Saved files in {model_dir}:")
for f in sorted(files):
    size_mb = os.path.getsize(os.path.join(model_dir, f)) / 1e6
    print(f"  {f:40s} {size_mb:.1f} MB")

## 6. Save model to Google Drive

Colab sessions are ephemeral — this copies the trained model to your Drive so it persists.

In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")

DRIVE_DEST = "/content/drive/MyDrive/rlhf-bias-decomp/results/reward_model_baseline"
os.makedirs(os.path.dirname(DRIVE_DEST), exist_ok=True)

shutil.copytree(model_dir, DRIVE_DEST, dirs_exist_ok=True)
print(f"Model saved to Google Drive: {DRIVE_DEST}")

## 7. Quick inference check

Sanity check: chosen response should score higher than rejected.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir).cuda()
model.eval()

def get_reward(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.cuda() for k, v in inputs.items()}
    with torch.no_grad():
        return model(**inputs).logits.squeeze().item()

test_ds = load_dataset("Anthropic/hh-rlhf", split="test")
correct = 0
for i in range(10):
    ex = test_ds[i]
    c, r = get_reward(ex["chosen"]), get_reward(ex["rejected"])
    correct += int(c > r)
    print(f"Ex {i+1}: chosen={c:+.3f}  rejected={r:+.3f}  {'✓' if c > r else '✗'}")

print(f"\nAccuracy on 10 examples: {correct}/10")